<a href="https://colab.research.google.com/github/baramitha/bt4222/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Data Loading

We'll be working with user ratings, anime details, and user details.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
user_filtered = pd.read_csv('/content/drive/My Drive/Colab Notebooks/user-filtered.csv.zip')
anime_filtered = pd.read_csv('/content/drive/My Drive/Colab Notebooks/anime-filtered.csv.zip')
users_details = pd.read_csv('/content/drive/My Drive/Colab Notebooks/users-details-2023.csv.zip')
print(f"User filtered shape: {user_filtered.shape}")
print(f"Anime filtered shape: {anime_filtered.shape}")
print(f"Users details shape: {users_details.shape}")

Mounted at /content/drive
User filtered shape: (109224747, 3)
Anime filtered shape: (14952, 25)
Users details shape: (731290, 16)


### Package Import

In [2]:
import numpy as np
import copy
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

### Data Cleaning

This section outlines the preprocessing steps taken to ensure data quality and consistency across our three main datasets:

#### 1. Anime Dataset (`anime_clean`)
- **Column Standardization**: Stripped whitespace and replaced spaces with underscores for easier programmatic access.
- **Type Casting & Missing Values**:
    - Converted `Score` and `Episodes` to numeric types, filling 'Unknown' or NaN values with `0`.
    - Filled missing `Genres` and `Type` values with 'Unknown'.
- **Deduplication**: Removed duplicate entries based on `anime_id` to ensure each title is unique.

#### 2. Users Details Dataset (`users_clean`)
- **Date Formatting**: Converted `Birthday` and `Joined` columns into standard datetime objects.
- **Numeric Cleanup**: Standardized columns like `Days_Watched`, `Mean_Score`, and `Episodes_Watched` to numeric formats, filling NaNs with `0`.

#### 3. Ratings Dataset (`ratings_clean`)
- **Explicit Feedback Filter**: Removed rows where the rating was `-1`. In this dataset, `-1` typically indicates the user added the anime to their list but did not provide an explicit score. Filtering these out focuses the model on explicit user preferences.

In [3]:
anime_clean = anime_filtered.copy()

# Standardize column names
anime_clean.columns = [col.strip().replace(' ', '_') for col in anime_clean.columns]
print(f"Original columns: {anime_clean.columns.tolist()}")

# Handle missing values
anime_clean['Score'] = pd.to_numeric(anime_clean['Score'], errors='coerce').fillna(0)
anime_clean['Episodes'] = anime_clean['Episodes'].replace('Unknown', 0)
anime_clean['Episodes'] = pd.to_numeric(anime_clean['Episodes'], errors='coerce').fillna(0)
anime_clean['Genres'] = anime_clean['Genres'].fillna('Unknown')
anime_clean['Type'] = anime_clean['Type'].fillna('Unknown')

# Remove duplicates
anime_clean = anime_clean.drop_duplicates(subset=['anime_id']).reset_index(drop=True)
print(f"Cleaned anime: {len(anime_clean):,} unique anime")

# 1.2 Clean Users Details Dataset
users_clean = users_details.copy()
users_clean.columns = [col.strip().replace(' ', '_') for col in users_clean.columns]

# Convert date columns
date_cols = ['Birthday', 'Joined']
for col in date_cols:
    if col in users_clean.columns:
        users_clean[col] = pd.to_datetime(users_clean[col], errors='coerce')

# Fill numeric columns
numeric_cols = ['Days_Watched', 'Mean_Score', 'Episodes_Watched']
for col in numeric_cols:
    if col in users_clean.columns:
        users_clean[col] = pd.to_numeric(users_clean[col], errors='coerce').fillna(0)

print(f"Cleaned users: {len(users_clean):,} users")

# 1.3 Clean Ratings Dataset
ratings_clean = user_filtered.copy()
ratings_clean.columns = ['user_id', 'anime_id', 'rating']

# Check for -1 ratings (watched but not rated)
print(f"Original ratings: {len(ratings_clean):,}")
neg_ratings = (ratings_clean['rating'] == -1).sum()
print(f"Ratings with -1: {neg_ratings:,} ({neg_ratings/len(ratings_clean)*100:.2f}%)")

# DECISION: Remove -1 ratings for cleaner explicit feedback
ratings_clean = ratings_clean[ratings_clean['rating'] != -1].copy()
print(f"After removing -1: {len(ratings_clean):,} ratings")


Original columns: ['anime_id', 'Name', 'Score', 'Genres', 'English_name', 'Japanese_name', 'sypnopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped']
Cleaned anime: 14,952 unique anime
Cleaned users: 731,290 users
Original ratings: 109,224,747
Ratings with -1: 0 (0.00%)
After removing -1: 109,224,747 ratings
